In [ ]:
import os
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import pipeline

def load_file(file_path):
    if file_path.lower().endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.lower().endswith(".txt"):
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError("Unsupported file type. Please provide a .pdf or .txt file.")
    return loader.load()

def main():
    file_path = input("Enter path to PDF or TXT file: ").strip()

    print("\nLoading document...")
    docs = load_file(file_path)

    print("Splitting text into chunks...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    print("Generating embeddings using HuggingFace...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)

    print("Loading Hugging Face LLM (gpt2)...")
    hf_pipeline = pipeline("text-generation", model="gpt2", max_new_tokens=100)
    llm = HuggingFacePipeline(pipeline=hf_pipeline)

    qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())

    print("\nReady! Ask your questions below.\n(Type 'exit' to quit.)")

    while True:
        query = input("Question: ")
        if query.lower() == "exit":
            break
        answer = qa.run(query)
        print("Answer:", answer)

if __name__ == "__main__":
    main()
